In [1]:
import sys
import gc
import importlib
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append("../src")

import models
import replay_buffer
import train
import evaluation

importlib.reload(models)
importlib.reload(replay_buffer)
importlib.reload(train)
importlib.reload(evaluation)

from models import DQN
from train import ConfigDQN, entrenar_dqn
from evaluation import evaluar_modelo

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print(f"Dispositivo: {device}")

Dispositivo: mps


# Ajuste de hiperparámetros de DQN + PER + n-step

En este notebook se analiza el efecto de dos hiperparámetros:

- El grado de priorización del replay buffer, controlado por `per_alpha`.
- El número de pasos utilizados para acumular recompensas, controlado por `n_step`.

Los demás hiperparámetros se mantendrán constantes para realizar una comparación controlada.

In [2]:
import gc

# Liberar modelos grandes que ya no necesitamos en memoria
objetos_temporales = [
    "resultado_dqn_per_3step",
    "mejor_dqn_per_3step",
    "checkpoint_3step",
]

for nombre in objetos_temporales:
    globals().pop(nombre, None)

gc.collect()

if device.type == "mps":
    torch.mps.empty_cache()

config_per_alpha04_3step = ConfigDQN(
    nombre_experimento="v7_dqn_per_a04_3step",
    total_pasos=1_000_000,

    seed=42,
    gamma=0.99,
    n_step=3,
    learning_rate=1e-4,
    batch_size=32,
    frecuencia_entrenamiento=4,

    capacidad_buffer=20_000,
    inicio_entrenamiento=10_000,
    frecuencia_actualizacion_target=10_000,
    gradient_clip=10.0,

    epsilon_inicial=1.0,
    epsilon_final=0.1,
    pasos_decay_epsilon=250_000,

    usar_per=True,
    per_alpha=0.4,
    per_beta_inicial=0.4,
    per_beta_final=1.0,
    per_pasos_beta=500_000,
    per_epsilon=1e-6,

    frecuencia_evaluacion=50_000,
    episodios_evaluacion=5,
    seed_evaluacion=1_000,
    frecuencia_log=1_000,

    terminal_on_life_loss=True,
    clip_reward=True,
)

resultado_per_alpha04_3step = entrenar_dqn(
    config=config_per_alpha04_3step,
    clase_modelo=DQN,
    device=device,
    usar_double_dqn=False,
)

print(
    "\nENTRENAMIENTO V7 FINALIZADO"
)

print(
    "Mejor promedio de evaluación: "
    f"{resultado_per_alpha04_3step['mejor_promedio_evaluacion']:.2f}"
)

A.L.E: Arcade Learning Environment (version 0.10.1+6a7e0ae)
[Powered by Stella]


Replay buffer: priorizado | alpha=0.4 | beta inicial=0.4
Retorno utilizado: 3-step
Paso 10,000/1,000,000 | episodio=49 | epsilon=0.964 | loss=0.0550 | Q=0.006
Paso 11,000/1,000,000 | episodio=53 | epsilon=0.960 | loss=0.0331 | Q=0.128
Paso 12,000/1,000,000 | episodio=59 | epsilon=0.957 | loss=0.0270 | Q=0.138
Paso 13,000/1,000,000 | episodio=62 | epsilon=0.953 | loss=0.0308 | Q=0.126
Paso 14,000/1,000,000 | episodio=67 | epsilon=0.950 | loss=0.0261 | Q=0.158
Paso 15,000/1,000,000 | episodio=71 | epsilon=0.946 | loss=0.0160 | Q=0.170
Paso 16,000/1,000,000 | episodio=77 | epsilon=0.942 | loss=0.0417 | Q=0.188
Paso 17,000/1,000,000 | episodio=81 | epsilon=0.939 | loss=0.0172 | Q=0.149
Paso 18,000/1,000,000 | episodio=84 | epsilon=0.935 | loss=0.0445 | Q=0.160
Paso 19,000/1,000,000 | episodio=88 | epsilon=0.932 | loss=0.0268 | Q=0.192
Paso 20,000/1,000,000 | episodio=92 | epsilon=0.928 | loss=0.0418 | Q=0.218
Paso 21,000/1,000,000 | episodio=97 | epsilon=0.924 | loss=0.0049 | Q=0.253
Paso 

In [3]:
ruta_evaluaciones_v7 = Path(
    "../logs/entrenamientos/"
    "v7_dqn_per_a04_3step/evaluaciones.csv"
)

df_evaluaciones_v7 = pd.read_csv(
    ruta_evaluaciones_v7
)

df_evaluaciones_v7 = (
    df_evaluaciones_v7
    .sort_values("paso_global")
    .reset_index(drop=True)
)

display(df_evaluaciones_v7)

mejor_evaluacion_v7 = df_evaluaciones_v7.loc[
    df_evaluaciones_v7["promedio"].idxmax()
]

ultimas_tres_v7 = (
    df_evaluaciones_v7
    .tail(3)["promedio"]
    .mean()
)

print("\nRESUMEN DE V7")
print(
    f"Mejor paso: "
    f"{int(mejor_evaluacion_v7['paso_global']):,}"
)
print(
    f"Mejor promedio: "
    f"{mejor_evaluacion_v7['promedio']:.2f}"
)
print(
    f"Mediana del mejor checkpoint: "
    f"{mejor_evaluacion_v7['mediana']:.2f}"
)
print(
    f"Desviación: "
    f"{mejor_evaluacion_v7['desviacion']:.2f}"
)
print(
    f"Mínimo: "
    f"{mejor_evaluacion_v7['minimo']:.2f}"
)
print(
    f"Máximo: "
    f"{mejor_evaluacion_v7['maximo']:.2f}"
)
print(
    f"Promedio de las últimas tres evaluaciones: "
    f"{ultimas_tres_v7:.2f}"
)

ruta_mejor_v7 = Path(
    "../models/v7_dqn_per_a04_3step/"
    "mejor_modelo.pt"
)

print(
    "\nCheckpoint disponible: "
    f"{'OK' if ruta_mejor_v7.exists() else 'NO ENCONTRADO'}"
)

,paso_global,promedio,mediana,desviacion,minimo,maximo
0,50000,149.0,135.0,52.478567,95.0,240.0
1,100000,191.0,160.0,116.206712,70.0,330.0
2,150000,297.0,295.0,116.730459,135.0,460.0
3,200000,383.0,305.0,123.393679,255.0,545.0
4,250000,302.0,305.0,80.659779,175.0,420.0
5,300000,242.0,160.0,113.604577,155.0,445.0
6,350000,250.0,200.0,85.029407,160.0,365.0
7,400000,478.0,465.0,124.923977,325.0,630.0
8,450000,267.0,220.0,113.428392,130.0,425.0
9,500000,294.0,330.0,103.121288,160.0,415.0



RESUMEN DE V7
Mejor paso: 1,000,000
Mejor promedio: 493.00
Mediana del mejor checkpoint: 480.00
Desviación: 114.04
Mínimo: 365.00
Máximo: 705.00
Promedio de las últimas tres evaluaciones: 438.67

Checkpoint disponible: OK


## Extensión de V7 hasta 1,500,000 pasos

Debido a que el mejor resultado de V7 ocurrió en el último checkpoint y las evaluaciones finales mostraron una tendencia favorable, se continúa el entrenamiento durante 500,000 pasos adicionales.

El replay buffer comienza vacío al reanudar, por lo que se recopilan 10,000 experiencias nuevas antes de retomar las actualizaciones.

In [4]:
import importlib
import gc

import train
importlib.reload(train)

from train import ConfigDQN, entrenar_dqn

ruta_checkpoint_v7 = Path(
    "../models/v7_dqn_per_a04_3step/"
    "checkpoint_final.pt"
)

if not ruta_checkpoint_v7.exists():
    raise FileNotFoundError(
        f"No se encontró el checkpoint: "
        f"{ruta_checkpoint_v7}"
    )

gc.collect()

if device.type == "mps":
    torch.mps.empty_cache()

config_v7_extendido = ConfigDQN(
    nombre_experimento=(
        "v7_dqn_per_a04_3step_extendido"
    ),
    total_pasos=1_500_000,

    seed=42,
    gamma=0.99,
    n_step=3,
    learning_rate=1e-4,
    batch_size=32,
    frecuencia_entrenamiento=4,

    capacidad_buffer=20_000,
    inicio_entrenamiento=10_000,
    frecuencia_actualizacion_target=10_000,
    gradient_clip=10.0,

    epsilon_inicial=1.0,
    epsilon_final=0.1,
    pasos_decay_epsilon=250_000,

    usar_per=True,
    per_alpha=0.4,
    per_beta_inicial=0.4,
    per_beta_final=1.0,
    per_pasos_beta=500_000,
    per_epsilon=1e-6,

    frecuencia_evaluacion=50_000,
    episodios_evaluacion=5,
    seed_evaluacion=1_000,
    frecuencia_log=1_000,

    terminal_on_life_loss=True,
    clip_reward=True,
)

resultado_v7_extendido = entrenar_dqn(
    config=config_v7_extendido,
    clase_modelo=DQN,
    device=device,
    usar_double_dqn=False,
    ruta_checkpoint_inicial=(
        ruta_checkpoint_v7
    ),
)

print(
    "\nEXTENSIÓN DE V7 FINALIZADA"
)

print(
    "Mejor promedio durante la extensión: "
    f"{resultado_v7_extendido['mejor_promedio_evaluacion']:.2f}"
)

Reanudando entrenamiento desde el paso 1,000,000 (checkpoint: ../models/v7_dqn_per_a04_3step/checkpoint_final.pt)
Replay buffer: priorizado | alpha=0.4 | beta inicial=0.4
Retorno utilizado: 3-step
Paso 1,010,000/1,500,000 | episodio=25 | epsilon=0.100 | loss=0.0399 | Q=5.046
Paso 1,011,000/1,500,000 | episodio=28 | epsilon=0.100 | loss=0.0203 | Q=4.656
Paso 1,012,000/1,500,000 | episodio=31 | epsilon=0.100 | loss=0.0062 | Q=5.263
Paso 1,013,000/1,500,000 | episodio=33 | epsilon=0.100 | loss=0.0217 | Q=5.342
Paso 1,014,000/1,500,000 | episodio=37 | epsilon=0.100 | loss=0.0265 | Q=5.254
Paso 1,015,000/1,500,000 | episodio=39 | epsilon=0.100 | loss=0.0082 | Q=5.395
Paso 1,016,000/1,500,000 | episodio=40 | epsilon=0.100 | loss=0.0164 | Q=4.702
Paso 1,017,000/1,500,000 | episodio=43 | epsilon=0.100 | loss=0.0022 | Q=5.224
Paso 1,018,000/1,500,000 | episodio=47 | epsilon=0.100 | loss=0.0056 | Q=4.788
Paso 1,019,000/1,500,000 | episodio=50 | epsilon=0.100 | loss=0.0056 | Q=4.933
Paso 1,020,00

In [5]:
ruta_evaluaciones_v7_ext = Path(
    "../logs/entrenamientos/"
    "v7_dqn_per_a04_3step_extendido/"
    "evaluaciones.csv"
)

df_evaluaciones_v7_ext = pd.read_csv(
    ruta_evaluaciones_v7_ext
)

df_evaluaciones_v7_ext = (
    df_evaluaciones_v7_ext
    .sort_values("paso_global")
    .reset_index(drop=True)
)

display(df_evaluaciones_v7_ext)

mejor_evaluacion_v7_ext = (
    df_evaluaciones_v7_ext.loc[
        df_evaluaciones_v7_ext[
            "promedio"
        ].idxmax()
    ]
)

print("\nMEJOR EVALUACIÓN DE V7 EXTENDIDO")
print(
    f"Paso: "
    f"{int(mejor_evaluacion_v7_ext['paso_global']):,}"
)
print(
    f"Promedio: "
    f"{mejor_evaluacion_v7_ext['promedio']:.2f}"
)
print(
    f"Mediana: "
    f"{mejor_evaluacion_v7_ext['mediana']:.2f}"
)
print(
    f"Desviación: "
    f"{mejor_evaluacion_v7_ext['desviacion']:.2f}"
)
print(
    f"Mínimo: "
    f"{mejor_evaluacion_v7_ext['minimo']:.2f}"
)
print(
    f"Máximo: "
    f"{mejor_evaluacion_v7_ext['maximo']:.2f}"
)

ruta_mejor_v7_ext = Path(
    "../models/"
    "v7_dqn_per_a04_3step_extendido/"
    "mejor_modelo.pt"
)

print(
    "\nCheckpoint disponible: "
    f"{'OK' if ruta_mejor_v7_ext.exists() else 'NO ENCONTRADO'}"
)

,paso_global,promedio,mediana,desviacion,minimo,maximo
0,1050000,455.0,450.0,122.759928,315.0,615.0
1,1100000,508.0,425.0,254.727305,285.0,990.0
2,1150000,475.0,475.0,60.580525,385.0,575.0
3,1200000,447.0,380.0,150.086642,345.0,745.0
4,1250000,462.0,470.0,92.336342,290.0,545.0
5,1300000,485.0,455.0,96.332757,350.0,600.0
6,1350000,427.0,325.0,197.651208,225.0,775.0
7,1400000,631.0,550.0,279.291962,340.0,1100.0
8,1450000,565.0,580.0,205.377701,330.0,885.0
9,1500000,465.0,375.0,149.432259,310.0,720.0



MEJOR EVALUACIÓN DE V7 EXTENDIDO
Paso: 1,400,000
Promedio: 631.00
Mediana: 550.00
Desviación: 279.29
Mínimo: 340.00
Máximo: 1100.00

Checkpoint disponible: OK


In [6]:
import evaluation
importlib.reload(evaluation)

from evaluation import evaluar_modelo
from models import DQN

N_EPISODIOS_EVALUACION = 30
SEMILLA_BASE_EVALUACION = 42

checkpoint_v7_ext = torch.load(
    ruta_mejor_v7_ext,
    map_location=device,
    weights_only=True,
)

mejor_v7_extendido = DQN(
    n_acciones=6
).to(device)

mejor_v7_extendido.load_state_dict(
    checkpoint_v7_ext[
        "modelo_online_state_dict"
    ]
)

mejor_v7_extendido.eval()

print(
    f"Checkpoint cargado desde el paso: "
    f"{checkpoint_v7_ext['paso']:,}"
)

print(
    "\nEvaluando V7 extendido durante "
    f"{N_EPISODIOS_EVALUACION} episodios...\n"
)

(
    resultados_v7_extendido,
    resumen_v7_extendido,
) = evaluar_modelo(
    modelo=mejor_v7_extendido,
    config=config_v7_extendido,
    device=device,
    n_episodios=N_EPISODIOS_EVALUACION,
    seed_base=SEMILLA_BASE_EVALUACION,
)

df_v7_extendido = pd.DataFrame(
    resultados_v7_extendido
)

df_v7_extendido["agente"] = (
    "DQN + PER a=0.4 + 3-step extendido"
)

df_v7_extendido = df_v7_extendido[
    [
        "agente",
        "episodio",
        "seed",
        "recompensa_total",
        "pasos",
        "terminated",
        "truncated",
    ]
]

print("RESUMEN DE 30 EPISODIOS")

for metrica, valor in resumen_v7_extendido.items():
    print(f"{metrica}: {valor:.2f}")

display(df_v7_extendido.head(10))

Checkpoint cargado desde el paso: 1,400,000

Evaluando V7 extendido durante 30 episodios...

RESUMEN DE 30 EPISODIOS
promedio: 421.83
mediana: 430.00
desviacion: 144.56
minimo: 180.00
maximo: 960.00


,agente,episodio,seed,recompensa_total,pasos,terminated,truncated
0,DQN + PER a=0.4 + 3-step extendido,0,42,630.0,1109,True,False
1,DQN + PER a=0.4 + 3-step extendido,1,43,400.0,726,True,False
2,DQN + PER a=0.4 + 3-step extendido,2,44,455.0,911,True,False
3,DQN + PER a=0.4 + 3-step extendido,3,45,280.0,645,True,False
4,DQN + PER a=0.4 + 3-step extendido,4,46,490.0,882,True,False
5,DQN + PER a=0.4 + 3-step extendido,5,47,330.0,613,True,False
6,DQN + PER a=0.4 + 3-step extendido,6,48,295.0,533,True,False
7,DQN + PER a=0.4 + 3-step extendido,7,49,485.0,880,True,False
8,DQN + PER a=0.4 + 3-step extendido,8,50,250.0,656,True,False
9,DQN + PER a=0.4 + 3-step extendido,9,51,960.0,1674,True,False


In [7]:
ruta_resultados_v7_ext = Path(
    "../logs/evaluacion_v7_extendido.csv"
)

df_v7_extendido.to_csv(
    ruta_resultados_v7_ext,
    index=False,
)

df_alpha06 = pd.read_csv(
    "../logs/evaluacion_dqn_per_3step.csv"
)

comparacion_alpha = (
    pd.concat(
        [
            df_alpha06,
            df_v7_extendido,
        ],
        ignore_index=True,
    )
    .pivot(
        index="seed",
        columns="agente",
        values="recompensa_total",
    )
    .dropna()
)

nombre_alpha06 = "DQN + PER + 3-step"
nombre_alpha04 = (
    "DQN + PER a=0.4 + 3-step extendido"
)

diferencia = (
    comparacion_alpha[nombre_alpha04]
    - comparacion_alpha[nombre_alpha06]
)

print(
    f"Victorias {nombre_alpha06}:",
    (diferencia < 0).sum(),
)

print(
    f"Victorias {nombre_alpha04}:",
    (diferencia > 0).sum(),
)

print(
    "Empates:",
    (diferencia == 0).sum(),
)

# Comparación del criterio mejor-de-cinco
df_competencia_alpha = pd.concat(
    [
        df_alpha06,
        df_v7_extendido,
    ],
    ignore_index=True,
)

df_competencia_alpha["bloque_de_5"] = (
    df_competencia_alpha["episodio"] // 5
) + 1

mejores_de_5_alpha = (
    df_competencia_alpha
    .groupby(
        ["agente", "bloque_de_5"],
        as_index=False,
    )
    .agg(
        mejor_recompensa=(
            "recompensa_total",
            "max",
        ),
        promedio_bloque=(
            "recompensa_total",
            "mean",
        ),
    )
)

resumen_competencia_alpha = (
    mejores_de_5_alpha
    .groupby("agente")
    .agg(
        mejor_de_5_promedio=(
            "mejor_recompensa",
            "mean",
        ),
        mejor_de_5_mediana=(
            "mejor_recompensa",
            "median",
        ),
        menor_mejor_de_5=(
            "mejor_recompensa",
            "min",
        ),
        mayor_mejor_de_5=(
            "mejor_recompensa",
            "max",
        ),
    )
    .round(2)
    .reset_index()
)

print(
    "\nCOMPARACIÓN DEL MEJOR DE CINCO"
)

display(resumen_competencia_alpha)

Victorias DQN + PER + 3-step: 17
Victorias DQN + PER a=0.4 + 3-step extendido: 12
Empates: 1

COMPARACIÓN DEL MEJOR DE CINCO


,agente,mejor_de_5_promedio,mejor_de_5_mediana,menor_mejor_de_5,mayor_mejor_de_5
0,DQN + PER + 3-step,611.67,582.5,490.0,770.0
1,DQN + PER a=0.4 + 3-step extendido,617.50,562.5,465.0,960.0
